# Open the Nigeria Demographic Explorer in Jupyter

Run the cell below. It starts a small local web server for the project folder (the folder that contains `index.html`) and shows the map here.

* Works in Jupyter Notebook, JupyterLab and VS Code notebooks.
* Needs internet the first time (map library and fonts).
* To stop the server, restart the kernel.

In [ ]:
import os, threading, functools, http.server, socketserver
from pathlib import Path
from IPython.display import IFrame, display, HTML

# The project folder is the parent of this notebooks/ folder. Change this if you moved the notebook.
PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (PROJECT / "index.html").exists(), f"index.html not found in {PROJECT}. Set PROJECT to the folder that contains it."

PORT = 8000
handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=str(PROJECT))

class QuietServer(socketserver.ThreadingTCPServer):
    allow_reuse_address = True
    daemon_threads = True

try:
    httpd = QuietServer(("127.0.0.1", PORT), handler)
    threading.Thread(target=httpd.serve_forever, daemon=True).start()
    print(f"Serving {PROJECT} at http://localhost:{PORT}")
except OSError:
    print(f"Port {PORT} is already in use; the server is probably running already.")

display(HTML(f'<a href="http://localhost:{PORT}/index.html" target="_blank">Open full screen in your browser</a>'))
IFrame(f"http://localhost:{PORT}/index.html", width="100%", height=820)

## Peek at the data behind the map (optional)

The cleaned tables are in `processed/` after you run the processing scripts. The web page itself reads the small files in `data/`.

In [ ]:
import json
profile = json.load(open(PROJECT / "data" / "state" / "NG020.json"))   # NG020 = Kano
print(profile["state"], "-", f'{profile["population"]["total"]:,}', "people")
print("Under 15:", profile["age_sex"]["pct_0_14"], "%")
print("Health facilities:", profile["health_facilities"]["by_level"])